# Indium BEC — benchmark run

Run top to bottom. **Only the "Knobs" cell changes between runs.**
Everything else is wiring you read once and leave alone.

## 0 · Setup (once per Colab session)

In [ ]:
import os

REPO = "/content/BECSimulation"
if not os.path.exists(REPO):
    !git clone -q https://github.com/svijaymurugan/BECSimulation.git {REPO}
else:
    !git -C {REPO} pull -q        # pick up anything you've pushed since
%cd {REPO}/newsim

# re-read edited .py files before every cell - must come BEFORE the imports
%load_ext autoreload
%autoreload 2

from google.colab import drive
drive.mount("/content/drive")

## 1 · Imports

In [1]:
import math, time, warnings
from pathlib import Path

import numpy as np
import torch
from IPython.display import display

from config import SimulationConfig
from grid import Grid
from cutoff import make_cutoff
from interactions import make_terms
from potentials import make_potential
from evolution import RealTimeEvolution, ImaginaryTimeEvolution
from diagnostics import Recorder, NormMonitor
from planning import grid_requirements
import storage, plotting

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

device: cpu 


## 2 · Knobs — the only cell you edit

In [3]:
cfg = SimulationConfig(
    # --- physics ---
    Np=1e5,                       # particle number
    m=1.9e-25,                    # atomic mass (kg)
    omega=2 * math.pi * 40,       # trap frequency (rad/s)
    a0=2e-9,                      # s-wave scattering length (m)
    a02=2e-23,                    # scattering volume, g2 term (m^3)

    # --- grid: sized with grid_requirements() below ---
    up=3e-5,                      # half box length (m): ~3 sigma
    N=400,                        # points in x and y
    Nz=None,                      # None = same as N
    sigma=10e-6,                  # Gaussian ansatz width (m)

    # --- trap ---
    gamma=1.0, imag_gamma=1.0,
    modulation_amp=0.0, modulation_freq=0.0,

    # --- real time ---
    ev_time=0.012,                # 12 ms: through the T/4 focus and back out
    time_steps=10_000,

    # --- imaginary time (only used when START_FROM = "ground_state") ---
    imag_dtau=1e-4, imag_max_steps=15_000, tolerance=1e-6,

    # --- numerics ---
    cutoff="Hard Cutoff",
    cutoff_kc=1e7,                 # PHYSICAL cutoff, m^-1 (from the reference)
    include_g0=True, include_g2=True,
    high_precision=True,

    # --- recording ---
    track_waist=True,
    track_modes=True,             # the kx/kz spectrogram shows aliasing directly
)

# --- choices about THIS run, not the physics, so they don't live in cfg ---
START_FROM    = "gaussian"        # "gaussian" (benchmark) or "ground_state" (cached ITE)
MEASURE_EVERY = 10                # 10,000 steps -> 1,000 energy samples, 12 us apart
FRAME_STRIDE  = 10                # a frame every 10th sample -> 100 GIF frames
SAVE_PSI      = False             # final 3-D psi is ~1 GB at N=400
LABEL         = "benchmark_N400"
DESCRIPTION   = ("Benchmark vs old code and the reference: Gaussian start, "
                 "sigma = 10 um, box +/-30 um, N = 400, physical cutoff 1e7 m^-1.")
OUT           = Path("/content/drive/MyDrive/BECSimulation/storage")

cfg.print_summary()

SimulationConfig

Physical
  particle number                100000
  atomic mass                   1.9e-25 kg
  trap frequency                     40 Hz       (omega = 251.3 rad/s)
  scattering length                   2 nm
  scattering volume               2e-23 m^3

Scales
  oscillator length l           1.48608 um
  trap period tau               3.97887 ms
  G0                            1691.21          contact, unitless
  G2                            25.6857          quadrupole, unitless

Grid
  points (N, N, nz)      400 x 400 x 400
  half box                           30 um       = 20.19 l
  spacing dx                   0.100937 l        OK
  ansatz sigma                       10 um       = 6.73 l

Real time
  duration                           12 ms
  steps                           10000
  dt                                1.2 us       = 0.0003016 tau
  z anisotropy gamma                  1
  modulation                        off          -> static trap, V built once

Imagina

## 3 · Pre-flight — is the grid good enough, and will it fit?

`grid_requirements` estimates the box and N from the cloud size, the focus
momentum and the cutoff. It's an estimate: the real test is running two N
and checking they agree.

In [4]:
N_min = grid_requirements(cfg)
if cfg.N < N_min:
    warnings.warn(f"N = {cfg.N} is below the estimate N >= {N_min}", RuntimeWarning)

# rough GPU memory: a step keeps ~8 complex arrays alive at once
if device == "cuda":
    need = 8 * cfg.N**2 * cfg.nz * 16 / 1e9
    have = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"memory:   ~{need:.1f} GB needed, {have:.1f} GB on this GPU"
          + ("   <-- WILL NOT FIT: lower N" if need > 0.9 * have else ""))

# what t=0 SHOULD show, so the first energy row can be checked against it
HZ = cfg.omega / (2 * math.pi)                 # hbar*omega expressed as E/h in Hz
V0 = 0.75 * (cfg.sigma / cfg.l)**2             # <V> of an untruncated Gaussian
print(f"expected <V>(t=0): {V0:.2f} hbar*omega = {V0 * HZ / 1e3:.3f} kHz   "
      f"(reference quoted 1.5 kHz = {1500 / HZ:.1f} hbar*omega)")

box:      up >= 30.0 um   (you have 30.0 um)  <-- TOO SMALL
momentum: focus needs 1.441e+07 m^-1, cutoff needs 1.500e+07 m^-1
grid:     dx <= 0.209 um  ->  N >= 288   (you have 400)  ~0.38 GB per complex array
expected <V>(t=0): 33.96 hbar*omega = 1.358 kHz   (reference quoted 1.5 kHz = 37.5 hbar*omega)


## 4 · Build — identical every run, don't edit

In [5]:
grid   = Grid(cfg, device)
cutoff = make_cutoff(cfg, grid)     # raises if the cutoff sits above the grid's k_max
terms  = make_terms(cfg, grid, cutoff)
# (if you've added the g2 ramp: make_terms(..., stage="imag") for ITE,
#  make_terms(..., stage="real") for real time)
names  = [t.name for t in terms]
print("terms:", names)

terms: ['g0 (contact)', 'g2 (quadrupole)']


## 5 · Initial state, and checks on it before spending GPU time

In [ ]:
if START_FROM == "gaussian":
    psi0 = grid.gaussian().to(device)

elif START_FROM == "ground_state":
    gs_path = storage.ground_state_path(cfg, root=OUT / "ground_states")
    if gs_path.exists():
        psi0, meta = storage.load_ground_state(gs_path, cfg, device)
        print(f"ground state from cache: E = {meta['final_energy']:.6f}, "
              f"{meta['steps']} steps, created {meta['created']}")
    else:
        rec_ite = Recorder.energies_only(names, dt=cfg.imag_dtau)
        ite = ImaginaryTimeEvolution(
            grid, make_potential(cfg, grid, cfg.imag_gamma), terms,
            dtau=cfg.imag_dtau, max_steps=cfg.imag_max_steps,
            recorder=rec_ite, tolerance=cfg.tolerance)
        psi0 = ite.run(grid.gaussian().to(device))
        storage.save_ground_state(gs_path, psi0, cfg, converged=ite.converged,
                                  final_energy=rec_ite.last_total(),
                                  steps=ite.steps_taken)
        print(f"ground state computed in {ite.steps_taken} steps, cached at {gs_path}")
else:
    raise ValueError(f"START_FROM must be 'gaussian' or 'ground_state', got {START_FROM!r}")

# --- checks ---
rho  = torch.abs(psi0)**2
edge = max(rho[0].max(), rho[:, 0].max(), rho[:, :, 0].max()).item() / rho.max().item()
V_t0 = (torch.sum(make_potential(cfg, grid, cfg.gamma)(0.0) * rho) * grid.dV).item()
del rho                              # 0.5 GB at N=400 - don't keep it around

print(f"norm:          {grid.unitless_norm(psi0).item():.12f}")
print(f"edge density:  {edge:.1e} of peak"
      + ("   OK" if edge < 1e-4 else "   <-- THE BOX TRUNCATES THE STATE"))
print(f"<V>(t=0):      {V_t0:.3f} hbar*omega   (expected {V0:.3f})")

xi = grid.healing_length(psi0)
if math.isfinite(xi):
    print(f"healing:       xi = {xi:.3f} l,  dx = {grid.dxu:.3f} l,  dx/xi = {grid.dxu / xi:.2f}")

## 6 · Real-time run

The Recorder is built **in this cell**, next to `run()`. Re-running the cell
always starts from an empty Recorder — building it in an earlier cell is what
produced the doubled energy plot.

In [ ]:
rec = Recorder(names,
               measure_every=MEASURE_EVERY, dt=cfg.dt,
               track_modes=cfg.track_modes,
               track_frames=True, frame_stride=FRAME_STRIDE,
               radius_squared=grid.radius_squared() if cfg.track_waist else None,
               dV=grid.dV)

rte = RealTimeEvolution(
    grid, make_potential(cfg, grid, cfg.gamma), terms,
    dtau=cfg.dtau, max_steps=cfg.time_steps,
    recorder=rec, measure_every=MEASURE_EVERY,
    monitors=[NormMonitor(grid)])

if device == "cuda": torch.cuda.synchronize()
t_start = time.perf_counter()
psi_final = rte.run(psi0)
if device == "cuda": torch.cuda.synchronize()
elapsed = time.perf_counter() - t_start

print(f"{rte.steps_taken} steps in {elapsed / 60:.1f} min "
      f"({1e3 * elapsed / rte.steps_taken:.1f} ms/step)")
print(f"final norm: {grid.unitless_norm(psi_final).item():.12f}")

## 7 · Save — to a fresh, timestamped directory on Drive

In [ ]:
run_dir  = storage.run_directory(root=OUT / "runs", label=LABEL, cfg=cfg)
run_path = storage.save_run(
    run_dir, cfg, grid, rec,
    description=f"{DESCRIPTION} [start: {START_FROM}]",
    psi_final=psi_final.cpu().numpy() if SAVE_PSI else None)
print("saved:", run_path)

## 8 · Plots — from the saved file, not the live Recorder

Loading back from disk proves the file is self-sufficient: this cell works
in any later session with no GPU, just by pointing `run_path` at an old run.

In [ ]:
cfg_dict, arrays = storage.load_run(run_path)

figs, gifs = plotting.standard_set(arrays, out_dir=run_dir / "plots",
                                   gif_fps=20, scale="power")
for name, fig in figs.items():
    display(fig)                                   # view first...
plotting.save_set(figs, run_dir / "plots")         # ...then save (this closes them)
print("GIFs:", *gifs.values(), sep="\n  ")

## 9 · Benchmark numbers

The handful of numbers to compare between this run, the old code, the
reference, and a lower-N convergence run.

In [ ]:
cols, E, t = list(arrays["columns"]), arrays["energies"], arrays["times"]
iV, iT, iG0 = cols.index("potential"), cols.index("total"), cols.index("g0 (contact)")
khz = HZ / 1e3

print(f"<V>(t=0):       {E[0, iV]:.3f} hbar*omega = {E[0, iV] * khz:.3f} kHz "
      f"(analytic {V0:.3f}; reference 1.5 kHz = {1500 / HZ:.1f})")

drift = abs(E[-1, iT] - E[0, iT]) / abs(E[0, iT])
print(f"energy drift:   {drift:.2e}")

k = int(np.argmax(E[:, iG0]))
print(f"focus:          peak g0 = {E[k, iG0]:.2f} at t = {t[k] * 1e3:.3f} ms "
      f"(T/4 = {1e3 * math.pi / (2 * cfg.omega):.3f} ms)")

print(f"end (t = {t[-1] * 1e3:.2f} ms): " +
      ", ".join(f"{c} = {E[-1, j]:.3f}" for j, c in enumerate(cols)))